# no_edge_bias

the Equation 9 additive edge term removed, leaving plain scaled dot-product attention over the same causal neighbourhood.

| | |
|---|---|
| **the one change** | ModelConfig.use_edge_bias True -> False |
| **answers** | reviewer comment 1 — the edge features are never ablated |
| **receptive field** | 421 nodes per target |
| **expected runtime** | ~8.7 h |
| **folds** | 17, weeks 8-24, seed 42, 200 epochs |

Run this notebook on its own, then **run the cleanup cell at the bottom yourself** and
shut the kernel down before opening the next one. Nothing frees the GPU automatically.

This arm is a NEW arm. It does not touch, retrain or overwrite any canonical or
protocol_v2 arm; `guards.py` enforces that four separate ways.


## 1. Free GPU memory right now

The one thing this notebook does automatically. Write the number down, or leave the output visible, so the cleanup at the bottom has something to compare against.

In [ ]:
import sys
from pathlib import Path

EK = Path.cwd() if Path.cwd().name == "edge_k" else Path.cwd().parent
assert (EK / "config.py").exists(), f"run this from edge_k/notebooks/, not {Path.cwd()}"

# ORDER MATTERS. Both edge_k/ and src/ contain a module called config.py — src's is the
# Sparkov preprocessing config, ours is the edge_k settings. Whichever directory comes
# first on sys.path wins, so edge_k is inserted LAST and therefore ends up FIRST.
for d in (str(EK.parent / "src"), str(EK.parent), str(EK)):
    while d in sys.path:
        sys.path.remove(d)
    sys.path.insert(0, d)

import config as CFG
assert Path(CFG.__file__).parent == EK, f"wrong config module: {CFG.__file__}"

import runner
free_at_start = runner.print_free_gpu("free at start")

## 2. Imports, canonical lock, frozen analysis

The canonical directories are checksummed **before** anything else happens, and again at the bottom.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import json
import guards, provenance as prov, controls, status as ST
import analysis_settings as ASET

# the canonical arms this package compares against, provably untouched
lock = guards.assert_canonical_unchanged()

# the analysis, frozen before any result exists (write-once; verified if already present)
alock = prov.write_analysis_lock()
print()
ASET.print_frozen()

## 3. What this arm is

Every setting is imported from `rerun_canonical.py` via `config.py`. Nothing is restated here.

In [ ]:
ARM = 'no_edge_bias'
CFG.print_summary(ARM)

## 4. POSITIVE CONTROLS — the hard gate

No training happens until the change is proved to have taken effect. A switch that is
not wired produces a clean null that reads as a finding.

This cell raises if any control fails. Do not work around it.

In [ ]:
gate = controls.run_gate(ARM)
print()
print("gate passed:", gate["passed"])

## 5. Load the frame

500,000 rows, 11 features, the 17 canonical rolling folds. Same construction as `rerun_canonical.build()`.

In [ ]:
B = runner.load_frame()

## 6. Graph control and preflight

Proves K actually changed the neighbour count, then checks every setting and projects peak VRAM on the largest fold. Raises rather than starting a run that will fail.

In [ ]:
graph = controls.check_graph(ARM, B)
print()
pf = runner.preflight(ARM, B)

## 7. Train

The only cell that costs GPU time. About **8.7 hours**.

`edge_k` never resumes. If this dies, move the arm directory to
`no_edge_bias_ABANDONED_<utc>`, note why in `STATUS.txt`, and start clean.

Progress is readable from another terminal while this runs:

```
tail -f edge_k/results/arms/no_edge_bias/STATUS.txt
```

In [ ]:
summary = runner.train_arm(ARM, B)
summary

## 8. CHECK — read-only, run this after training

Re-checks the canonical lock and prints the result against both declared baselines. Writes nothing.

In [ ]:
guards.assert_canonical_unchanged()

import pandas as pd
res = pd.read_csv(CFG.ARMS_DIR / ARM / "per_fold.csv")
mean_ap = res["ap"].mean()
for name, b in ASET.BASELINES[ARM].items():
    base = ASET.BASELINE_VALUES[b]["value"]
    d = mean_ap - base
    verdict = ("inside the resolution of a single execution"
               if abs(d) < ASET.THRESHOLDS["decision_threshold"] else "resolvable")
    print(f"{name:10} vs {b:16} {base:.4f}   arm {mean_ap:.4f}   "
          f"delta {d:+.4f}   {verdict}")
print()
print(f"threshold {ASET.THRESHOLDS['decision_threshold']}  "
      f"({ASET.THRESHOLDS['decision_basis']})")
print(ASET.THRESHOLDS["report_as"])

## 9. CLEANUP — run this yourself

**Not automatic, and deliberately so.** The model stays on the card until you say
otherwise, so it can be inspected after a run.

Run this, compare the freed memory against cell 1, then shut the kernel down before
opening the next notebook. Skipping it makes the next arm die partway through on a
memory error.

In [ ]:
import gc, torch
for name in ("summary", "B", "res"):
    if name in dir():
        del globals()[name]
gc.collect()
torch.cuda.empty_cache()
free_at_end = runner.print_free_gpu("free at end")
if free_at_start[0] and free_at_end[0]:
    print(f"  recovered {(free_at_end[0] - free_at_start[0])/1e9:+.1f} GB "
          f"relative to the start of this notebook")
print()
print("Now shut the kernel down before opening the next notebook.")